# Does a more flexible model do better?

Everything so far has used logistic regression. Here, two tree models are tested on the same
folds before committing:

- **Logistic regression** adds up each column's effect and draws one straight dividing line.
  It can't see "these two things *together*" unless someone builds that column.
- **Random forest** averages hundreds of decision trees, each trained on a different sample.
- **Gradient boosting** grows trees one after another, each fixing the last one's mistakes.

Trees can learn combinations, so this is also a second test of synergy: **if agent
combinations matter, a tree should find them.**

| Column set | Columns | What it is |
|---|---|---|
| Without player stats | 39 | agents, map, who picked (notebook 04) |
| With player stats | 2 | rating difference, who picked (notebook 06) |
| Everything | 41 | both |

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression

sys.path.insert(0, str(Path.cwd().parent / "src"))

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

from dataset import build_map_table, season_folds, split_season
from features import build_features, feature_columns
from form import player_form
from model import evaluate, make_model, picker_baseline, summarise

maps = build_map_table()
learn, test = split_season(maps)

X_agents, y = build_features(learn, feature_columns(maps))
X = pd.concat([X_agents, player_form(learn, half_life=5)], axis=1)
agent_columns = [c for c in X_agents.columns if c.startswith("agent_")]
map_columns = [c for c in X_agents.columns if c.startswith("map_") and c != "map_picked_by_a"]

SETS = {
    "without player stats (39)": list(X_agents.columns),
    "with player stats (2)": ["rating_diff", "map_picked_by_a"],
    "everything (41)": list(X_agents.columns) + ["rating_diff"],
}

baseline = picker_baseline(y, learn)
print(f"learning half: {len(learn):,} maps   (test half and 2026 not touched)")
print(f"baseline to beat: {baseline['accuracy'].mean():.1%}")
print("log loss for saying 50/50 to every map: 0.693")

learning half: 756 maps   (test half and 2026 not touched)
baseline to beat: 55.5%
log loss for saying 50/50 to every map: 0.693


## Checking the machinery

Every model goes through `src/model.py`: fill gaps, scale, fit, all inside each fold using
its training part only. First, it must reproduce the earlier hand-written notebooks exactly.

In [2]:
lr = make_model(LogisticRegression(max_iter=5000))
for name, expected in [("without player stats (39)", "51.1% / 0.760"),
                       ("with player stats (2)", "59.7% / 0.678")]:
    s = summarise(evaluate(lr, X[SETS[name]], y, learn), baseline)
    print(f"{name:28} {s['accuracy']:.1%} / {s['log loss']:.3f}    notebooks 04-06 got {expected}")

without player stats (39)    51.1% / 0.760    notebooks 04-06 got 51.1% / 0.760
with player stats (2)        59.7% / 0.678    notebooks 04-06 got 59.7% / 0.678


Identical, so the refactor changed nothing.

## 1. All three models, on default settings

The tree models use a fixed seed (`random_state=0`), so results repeat exactly.

In [3]:
FAMILIES = {
    "logistic regression": lambda: LogisticRegression(max_iter=5000),
    "random forest": lambda: RandomForestClassifier(random_state=0),
    "gradient boosting": lambda: HistGradientBoostingClassifier(random_state=0),
}

rows = []
for set_name, cols in SETS.items():
    for family, build in FAMILIES.items():
        s = summarise(evaluate(make_model(build()), X[cols], y, learn), baseline)
        rows.append({"columns": set_name, "model": family, **s})

defaults = pd.DataFrame(rows)
shown = defaults.copy()
for c in ["accuracy", "worst fold", "best fold"]:
    shown[c] = (shown[c] * 100).round(1)
shown["log loss"] = shown["log loss"].round(3)
print(shown.to_string(index=False))

                  columns               model  accuracy  worst fold  best fold  log loss  beats baseline
without player stats (39) logistic regression      51.1        47.3       54.3     0.760               2
without player stats (39)       random forest      52.4        45.7       56.6     0.774               1
without player stats (39)   gradient boosting      52.9        44.0       59.5     0.773               2
    with player stats (2) logistic regression      59.7        56.5       64.1     0.678               4
    with player stats (2)       random forest      52.8        47.6       59.0     1.635               2
    with player stats (2)   gradient boosting      52.1        48.4       58.8     0.736               0
          everything (41) logistic regression      53.9        50.4       62.9     0.747               2
          everything (41)       random forest      53.6        45.7       60.3     0.725               2
          everything (41)   gradient boosting      53.8

**Only one of the nine beats the baseline: logistic regression on two columns.** The trees
fall below the baseline on the two-column set: with 756 maps, the right shape is a straight
line, and trees learn noise instead.

**Random forest's log loss is 1.635**, more than double the 0.693 for knowing nothing:

In [4]:
for family in ["logistic regression", "random forest"]:
    chances, truths = [], []
    for learn_rows, check_rows in season_folds(learn):
        cols = SETS["with player stats (2)"]
        fitted = clone(make_model(FAMILIES[family]())).fit(
            X.loc[learn_rows, cols].astype(float), y.loc[learn_rows])
        chances.append(fitted.predict_proba(X.loc[check_rows, cols].astype(float))[:, 1])
        truths.append(y.loc[check_rows].to_numpy())
    p, t = np.concatenate(chances), np.concatenate(truths)

    sure = (p <= 0.05) | (p >= 0.95)
    sure_and_wrong = sure & ((p >= 0.5).astype(int) != t)
    print(f"{family}:")
    print(f"   maps it was 95%+ sure about : {sure.mean():.0%}")
    print(f"   ...and got wrong            : {sure_and_wrong.sum()}")
    print(f"   its predictions ranged from : {p.min():.2f} to {p.max():.2f}\n")

logistic regression:
   maps it was 95%+ sure about : 0%
   ...and got wrong            : 0
   its predictions ranged from : 0.23 to 0.78



random forest:
   maps it was 95%+ sure about : 23%
   ...and got wrong            : 66
   its predictions ranged from : 0.00 to 1.00



**Random forest is confidently wrong:** 95%+ sure on 23% of maps and wrong on 66 of them.
Logistic regression never goes past 78%. Default trees grow until each leaf holds one map, so
they memorise. Accuracy calls it mediocre; log loss shows it would mislead.

## 2. Giving the trees a fair shot

Only the settings that control over-confidence are tuned, 12 combinations each:
`min_samples_leaf` and `max_depth` for random forest, plus `learning_rate` for gradient
boosting. The trees get twelve tries and logistic regression one, untuned, which deliberately
favours the trees.

In [5]:
from itertools import product

GRIDS = {
    "random forest": [
        dict(min_samples_leaf=leaf, max_depth=depth)
        for leaf, depth in product([1, 10, 30, 60], [None, 3, 6])
    ],
    "gradient boosting": [
        dict(max_depth=depth, min_samples_leaf=leaf, learning_rate=rate)
        for depth, leaf, rate in product([None, 2, 3], [20, 50], [0.1, 0.03])
    ],
}
BUILD = {"random forest": RandomForestClassifier,
         "gradient boosting": HistGradientBoostingClassifier}

tuned = []
for set_name, cols in SETS.items():
    for family, grid in GRIDS.items():
        for params in grid:
            s = summarise(evaluate(make_model(BUILD[family](random_state=0, **params)),
                                   X[cols], y, learn), baseline)
            tuned.append({"columns": set_name, "model": family, "setting": params, **s})
tuned = pd.DataFrame(tuned)

lr_by_set = defaults[defaults["model"] == "logistic regression"].set_index("columns")

for set_name in SETS:
    lr_acc = lr_by_set.loc[set_name, "accuracy"]
    print(f"=== {set_name}    logistic regression, untuned: {lr_acc:.1%}")
    for family in GRIDS:
        f = tuned[(tuned["columns"] == set_name) & (tuned["model"] == family)]
        best = f.loc[f["accuracy"].idxmax()]
        print(f"   {family:18} {len(f)} settings, {f['accuracy'].min():.1%} to {f['accuracy'].max():.1%}"
              f"   | beat logistic regression: {(f['accuracy'] > lr_acc).sum():2}"
              f"   | log loss {f['log loss'].min():.3f} to {f['log loss'].max():.3f}")
        print(f"   {'':18} best {best['accuracy']:.1%} with {best['setting']}")
    print()

=== without player stats (39)    logistic regression, untuned: 51.1%
   random forest      12 settings, 51.6% to 53.6%   | beat logistic regression: 12   | log loss 0.691 to 0.774
                      best 53.6% with {'min_samples_leaf': 10, 'max_depth': 6}
   gradient boosting  12 settings, 52.5% to 56.5%   | beat logistic regression: 12   | log loss 0.697 to 0.773
                      best 56.5% with {'max_depth': 3, 'min_samples_leaf': 50, 'learning_rate': 0.1}

=== with player stats (2)    logistic regression, untuned: 59.7%
   random forest      12 settings, 52.0% to 58.2%   | beat logistic regression:  0   | log loss 0.673 to 1.635
                      best 58.2% with {'min_samples_leaf': 30, 'max_depth': None}
   gradient boosting  12 settings, 52.1% to 57.6%   | beat logistic regression:  0   | log loss 0.674 to 0.736
                      best 57.6% with {'max_depth': 3, 'min_samples_leaf': 50, 'learning_rate': 0.03}

=== everything (41)    logistic regression, untuned: 53.

**On the two columns that matter, none of the 24 tree settings beats untuned logistic
regression.** The best comes within a point and a half. Requiring 30 maps per leaf fixes
random forest's over-confidence (log loss 1.635 → 0.673), but it stays less accurate.

On the agent-heavy sets, nearly every tree beats logistic regression. Are they finding the
agents, or just the picker rule?

## 3. Agents, or the picker?

Refit the best tree with pieces taken away:

In [6]:
best_setting = (tuned[(tuned["columns"] == "without player stats (39)")
                      & (tuned["model"] == "gradient boosting")]
                .sort_values("accuracy").iloc[-1]["setting"])
print(f"best gradient-boosting setting on the agent set: {best_setting}\n")

pieces = {
    "agents + maps + who picked (all 39)": list(X_agents.columns),
    "maps + who picked, no agents": map_columns + ["map_picked_by_a"],
    "who picked only": ["map_picked_by_a"],
    "agents only, no who-picked": agent_columns,
}
for name, cols in pieces.items():
    s = summarise(evaluate(make_model(HistGradientBoostingClassifier(random_state=0, **best_setting)),
                           X[cols], y, learn), baseline)
    print(f"   {name:38} {s['accuracy']:.1%}   beats baseline on {s['beats baseline']}/5 folds")

best gradient-boosting setting on the agent set: {'max_depth': 3, 'min_samples_leaf': 50, 'learning_rate': 0.1}



   agents + maps + who picked (all 39)    56.5%   beats baseline on 3/5 folds


   maps + who picked, no agents           55.3%   beats baseline on 1/5 folds


   who picked only                        55.5%   beats baseline on 0/5 folds


   agents only, no who-picked             49.0%   beats baseline on 0/5 folds


**The tree's score is almost all the who-picked rule.** Without the agents it barely moves;
with the agents alone, a model able to learn their combinations scores **below a coin flip**.
The agents add about a point, inside the margin of error and from the best of twelve settings.

Scrambling one column at a time instead shows what the fitted model *leans on*:

In [7]:
cols = list(X_agents.columns)
leaning = pd.Series(0.0, index=cols)
for learn_rows, check_rows in season_folds(learn):
    fitted = clone(make_model(HistGradientBoostingClassifier(random_state=0, **best_setting))).fit(
        X.loc[learn_rows, cols].astype(float), y.loc[learn_rows])
    result = permutation_importance(fitted, X.loc[check_rows, cols].astype(float),
                                    y.loc[check_rows], n_repeats=10, random_state=0)
    leaning += pd.Series(result.importances_mean, index=cols) / 5

print("accuracy lost when one column is scrambled, averaged over the folds:\n")
for column, lost in leaning.sort_values(ascending=False).head(5).items():
    print(f"   {column:20} {lost * 100:+.2f} points")
print(f"\n   all 27 agent columns added together: {leaning[agent_columns].sum() * 100:+.2f} points")

accuracy lost when one column is scrambled, averaged over the folds:

   map_picked_by_a      +4.52 points
   agent_tejo           +1.26 points
   agent_vyse           +1.13 points
   agent_breach         +1.03 points
   agent_neon           +0.84 points

   all 27 agent columns added together: +6.69 points


Who picked is the column it leans on hardest; each agent is worth about a point. Together the
agents look bigger, but that's no contradiction. Scrambling shows what *this* model depends
on; refitting without a column shows what it *adds*. Refitting says the agents add about a
point: noise.

**That's notebook 07's answer reached a second, independent way.**

## 4. The decision

**Logistic regression on two columns, the players' rating difference and who picked the map,
is the final model.**

- **Most accurate:** none of 24 tuned tree settings beats it.
- **Honest probabilities:** log loss 0.678 against 0.693. Tuned trees match that (0.673) but
  are less accurate.
- **Easiest to explain:** two inputs, one weight each.
- **Hardest to fool yourself with:** almost nothing to tune.

## What this notebook found

1. Only one of nine combinations beats the baseline on defaults.
2. Trees lose on the two columns that matter, even tuned in their favour.
3. Default random forest is confidently wrong, which log loss exposed.
4. Trees are better at *ignoring* useless columns, which isn't the same as finding value in
   them.
5. Agent combinations add nothing this data can detect: two methods, one conclusion.